In [1]:
import importlib.util, subprocess, sys

_pkgs = ["langgraph", "langchain_community", "langchain_openai", "langsmith", "langgraph-swarm"]
_missing = [p for p in _pkgs if importlib.util.find_spec(p.replace("-", "_").split("[")[0]) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _pkgs)

In [2]:
# Environment Variable Initialization

import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from the notebook's working directory

def _set_if_undefined(var_name: str):
    value = os.environ.get(var_name, "").strip()
    if value:
        masked = value[:6] + "*" * (min(20, len(value) - 6))
        print(f"  ✅ {var_name}: {masked}")
    else:
        print(f"  ❌ {var_name}: not set")

# ---- Environment Variables Required ----

print("Checking environment variables...")
_set_if_undefined("OPENAI_API_KEY")         # API key for OpenAI models
_set_if_undefined("LANGSMITH_TRACING")      # Enable LangSmith tracing ("true" to enable)
_set_if_undefined("LANGSMITH_API_KEY")      # https://docs.langchain.com/langsmith/observability
_set_if_undefined("OPENAI_MODEL")           # e.g., "gpt-4.1", "gpt-4o", "gpt-4o-mini"
print("Done.")

Checking environment variables...
  ✅ OPENAI_API_KEY: sk-pro********************
  ✅ LANGSMITH_TRACING: true
  ✅ LANGSMITH_API_KEY: lsv2_p********************
  ✅ OPENAI_MODEL: gpt-4o*****
Done.


In [3]:
# Swarm Example:
# In this setup, each agent can communicate with every other agent (many-to-many connections).
# Agents can decide which agent to call next during their reasoning process.

import os, warnings
from typing import Annotated
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.warnings import LangGraphDeprecatedSinceV10
from langgraph_swarm import create_handoff_tool, create_swarm

# Silence the create_react_agent V1.0 deprecation warning so demo output is clean.
warnings.filterwarnings("ignore", category=LangGraphDeprecatedSinceV10)


# ---- Tool Definitions ----

# Define a dummy flight_information_tool
@tool
def flight_information_tool(bookingnumber: Annotated[str, "flight booking number"]):
    """Tool to fetch flight information for a given booking number."""
    # (Simulated API call — replace with real API integration)
    json_data = {
        "flightinfo": {
            "bookingnumber": bookingnumber,
            "departure_airport": "ARN" ,
            "arrival_airport": "DXB",
            "departure_time": "13:40",
            "departure_date": "Tuesday, 20 May 2025",
            "arrival_time": "19:50",
            "arrival_date": "Tuesday, 20 May 2025",
            "departure_gate": "Gate F58",
            "upgrade_availability": "Business Class is available for upgrade.",
            "status": "on-time"
        }
    }
    return json_data

# Load the model name from environment variables
openai_model = os.environ["OPENAI_MODEL"]
# Initialize the LLM (Large Language Model) interface
llm = ChatOpenAI(model=openai_model)


# ---- Agent Definitions ----

# Create the 'Viktor' agent:
# - Expert in flight related queries.
# - Can either solve the query directly or hand it off to Walter if necessary.
viktor  = create_react_agent(
    llm,
    tools=[
        flight_information_tool, 
        create_handoff_tool(agent_name="Walter", description="Transfer to Walter, he can help with luggage related queries")  # Tool to transfer the conversation 
    ],
    prompt=(
        "You are an Airline Support Agent (Viktor) designed to assist customers with flight-related queries. "
        "Your role is to provide accurate and relevant information about flights, gate changes, upgrades, and related services. "
        "You have access to a tool to fetch flight data, use the tool whenever required, do NOT guess or make up an answer."
        "Make sure to provide helpful, friendly responses, even if the information is not available."
        "Please keep going until the user’s query is completely resolved before ending your turn and yielding back to the user."
        "Only terminate your turn when you are sure that the query is answered or you have transferred the query to other agents."
    ),
    name="Viktor",
)



# Define a dummy luggage_support_tool
@tool
def luggage_information_tool(bookingnumber: Annotated[str, "flight booking number"]):
    """Tool to fetch luggage information for a given booking number."""
    # (Simulated API call — replace with real API integration)
    json_data = {
        "luggageinfo": {
            "bookingnumber": bookingnumber,
            "belt": "7" ,
            "limit": "30KGs",
            "lost": False,
            "delayed": False,
            "status": "on-belt"
        }
    }
    return json_data

# Create the 'Walter' agent:
# - Expert in luggage related queries.
# - He can only transfer the query back to Viktor if needed.
walter = create_react_agent(
    llm,
    tools=[
        luggage_information_tool,
        create_handoff_tool(agent_name="Viktor", description="Transfer to Viktor, he can help with flight related queries")
    ],
    prompt=(
        "You are an Luggage Support Agent (Walter) designed to assist customers with inquiries regarding luggage, baggage claims, and related services."
        "Your role is to provide accurate and relevant information about luggage limits, baggage belts, lost luggage, and luggage delays."
        "You have access to a tool to fetch luggage data, use the tool whenever required, do NOT guess or make up an answer."
        "Make sure to provide helpful, friendly responses, even if the information is not available."
        "Please keep going until the user’s query is completely resolved before ending your turn and yielding back to the user."
        "Only terminate your turn when you are sure that the query is answered or you have transferred the query to other agents."
    ),
    name="Walter",
)

In [4]:
# ---- Swarm and Workflow Definition ----

# In-memory checkpointing to save intermediate agent states during the conversation
checkpointer = InMemorySaver()
# long-term memory
store = InMemoryStore()
# Create a swarm (multi-agent environment) where agents can call each other
workflow = create_swarm(
    [viktor, walter],  # List of agents
    default_active_agent="Walter"  # Walter starts the conversation
)

# Compile the swarm workflow into an executable app
app = workflow.compile(checkpointer=checkpointer, store=store, name="travel_support")

In [5]:
# ---- Visualization ----
app.get_graph().print_ascii()

        +-----------+     
        | __start__ |     
        +-----------+     
          .         ..    
        ..            .   
       .               .. 
+--------+               .
| Viktor |             .. 
+--------+            .   
          .         ..    
           ..     ..      
             .   .        
          +--------+      
          | Walter |      
          +--------+      


In [6]:
# ---- Stream User Interactions ----

# Configuration for conversation (e.g., thread ID to track session)
config = {"configurable": {"thread_id": "11"}}

# --- Turn 1: User asks a flight related question ---

for chunk in app.stream(
    input={"messages": [{"role": "user", "content": "Whats the departure gate for my flight, booking number is ABC1234"}]},
    config=config,
    debug=True,
    subgraphs=True, 
    stream_mode="debug"
):
    print(chunk)
    print("============================")

((), {'step': -1, 'timestamp': '2026-09-19T17:53:12.050249+00:00', 'type': 'checkpoint', 'payload': {'config': {'configurable': {'checkpoint_ns': '', 'thread_id': '11', 'checkpoint_id': '1f1b452f-b3b9-602a-bfff-13e6b8775aeb'}}, 'parent_config': None, 'values': {'messages': []}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': '3bdd9427-9daf-307f-9446-f33c88ffd474', 'name': '__start__', 'interrupts': (), 'state': None}]}})
[values] {'messages': [HumanMessage(content='Whats the departure gate for my flight, booking number is ABC1234', additional_kwargs={}, response_metadata={}, id='85da7426-719f-4c61-9354-76184bc1c5b9')]}
((), {'step': 0, 'timestamp': '2026-09-19T17:53:12.051609+00:00', 'type': 'checkpoint', 'payload': {'config': {'configurable': {'checkpoint_ns': '', 'thread_id': '11', 'checkpoint_id': '1f1b452f-b3bc-6d42-8000-3461342f361b'}}, 'parent_config': {'configurable': {'checkpoint_ns': '', 'thread_id': '11', 'checkpoint_id': '1

In [7]:
# --- Turn 2: User asks a luggage related question ---

for chunk in app.stream(
    input={"messages": [{"role": "user", "content": "What's my luggage limit?"}]},
    config=config,
    debug=True,
    subgraphs=True, 
    stream_mode="debug"
):
    print(chunk)
    print("============================")

((), {'step': 3, 'timestamp': '2026-09-19T17:53:14.315691+00:00', 'type': 'checkpoint', 'payload': {'config': {'configurable': {'checkpoint_ns': '', 'thread_id': '11', 'checkpoint_id': '1f1b452f-c953-6b7e-8003-2db7ca41f5dd'}}, 'parent_config': {'configurable': {'checkpoint_ns': '', 'thread_id': '11', 'checkpoint_id': '1f1b452f-c949-6eee-8002-b118f8f8914c'}}, 'values': {'messages': [HumanMessage(content='Whats the departure gate for my flight, booking number is ABC1234', additional_kwargs={}, response_metadata={}, id='85da7426-719f-4c61-9354-76184bc1c5b9'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 226, 'total_tokens': 239, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 